In [ ]:
import seaborn as sns
import numpy as np
import glob
import matplotlib.pyplot as plt
import pprint
import matplotlib.pyplot as plt
from itertools import combinations
from  pathlib import Path
from readerscf import parse_sdr_file
from get_project_root import get_project_root
from replace_outliers import replace_outliers
from calculate_inverse_matrix import calculate_inverse_matrix
from detrend import detrend_dataframe,detrend_preserve_mean_level,detrend_and_shift_positive,detrend_preserve_baseline,detrend_full
from substractdf import subtract_column_min,subtract_percentile_norm
from multiply_matrix_with_dataframe import multiply_matrix_with_dataframe,matrix_multiply_with_dataframe
from subtract_mean_from_first_n import subtract_mean_from_first_n
from remove_baseline import remove_baseline
from config import IUPAC, ref_str, color_map
from center_dataframe import center_dataframe
from multiply_matrix_with_dataframe import multiply_matrix_with_dataframe

def main():
    project_root = get_project_root()
    project_root = Path(get_project_root())
    print(f"📁 Корень проекта: {project_root}")
    srd_path = project_root / "files" / "сырые SRD" / "сырые SRD" / "6_pGEM_4_G3_PDMA6_36.srd"
    print(srd_path)
    sdr_matrix, sdr_channels, sdr_meta = parse_sdr_file(srd_path)
    print(type(sdr_matrix))
    matrix=sdr_matrix.to_numpy()
    print(matrix)
    print(matrix.shape)
    matrix=matrix[6:10,0:4]
    print('marix')
    print(matrix)
    print('matrix inv')
    print(np.linalg.inv(matrix))
    # print(sdr_channels.head())
    data0=sdr_channels.loc[:, ['dR110', 'dR6G', 'dTAMRA', 'dROX']]
    new_columns=['G','A','T','C']
    data0.columns=new_columns
    

    colors = [color_map[col] for col in data0.columns] 
    data0.plot(color=colors)
    plt.suptitle('Raw before matrix')
    plt.grid(True)
    data0=subtract_column_min(data0)
    data0.plot(color=colors)
    plt.xlim(800,1500)
    plt.suptitle('Raw subtract min columns')
    plt.grid(True)
    data0= subtract_percentile_norm(data0, q=1.0)
    data0.plot(color=colors)
    plt.suptitle(f'Raw subtract percentile columns q={0.1}')
    plt.grid(True)
    data0d=detrend_dataframe(data0)
    data0d.plot(color=colors)
    plt.suptitle('Raw detrend ')
    plt.grid(True)
    data0d=detrend_preserve_baseline(data0)
    data0d.plot(color=colors)
    plt.suptitle('Raw detrend linear')
    plt.grid(True)
    data0d=detrend_full(data0)
    data0d.plot(color=colors)
    plt.suptitle('Raw detrend polynomal')
    plt.grid(True)
    data0cb,baselines= remove_baseline(data0,methods=["asls", "airpls", "imodpoly","poly"],poly={"poly_order":1},asls={"lam": 1e-1, "p": 0.001},airpls={"lam": 1e-1},imodpoly={"poly_order":1} )
    data0cb.plot(color=colors)
    plt.suptitle('Raw corrected baselines')
    plt.grid(True)

    baselines.plot(color=colors)
    plt.suptitle('Raw  baselines')
    plt.grid(True)


    data0c=center_dataframe(data0,method='percentile', percentile=2)
    data0c.plot(color=colors)
    plt.suptitle('Raw before matrix center')
    plt.grid(True)
    data1=multiply_matrix_with_dataframe(matrix,center_dataframe(data0,method='percentile', percentile=2))
    plt.show()
    
    
    
   
    
if __name__ == "__main__":
    main()